<table style="width:100%; border:2px solid #2583d8; border-radius:10px; background-color:white;">
<tr>

<td style="width:34%; vertical-align:middle; padding:20px;">
<img src="./assets/RoboDedito.png"
     alt="Robot character"
     style="width:100%; max-width:480px; display:block; margin:auto; border-radius:10px;">
</td>

<td style="width:66%; vertical-align:middle; padding:25px 35px;">

<h2 style="color:#1474c4; font-size:30px;">
Comparing Models.
</h2>

<p style="font-size:20px;"> Create a new folder for this project.</p>

<p style="font-size:20px;"> Open the folder using <code>VSCode<code>.</p>

<p style="font-size:20px;"> The uv package was already installed on the local computer (see previous lab)</p>

<p style="font-size:20px;"> Create a a virtual environment. Run the command <code>uv init</code> to start <code>uv<code></p>

<p style="font-size:20px;"> Create another folder named <code>.envme</code></p>

<p style="font-size:20px;"> Inside this folder, place a file called <code>Profile.pdf</code> containing your LinkedIn information. Place another file named <code>summary.txt</code> containing details about your persona. </p>

</td>
</tr>
</table>

## Comparing Models

This second project is about asking several LLMs the same question, collecting their answers,

and then using another LLM as a judge to rank the responses.

The first LLM will be OpenAI, and the rest will be several open-source models that can run in Ollama.

I will also use OpenAI to judge the answers.

First, I will import the necessary library modules



In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
# Always remember to do this!
load_dotenv(override=True)

In [ ]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

Next, I will ask OpenAI a question to evaluate the LLM's models

In [ ]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

The question will be generated using the model "gpt-5-mini"

In [ ]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)

Next, I will convert the answer from OpenAI in the question

In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

The next part will be the answer generation by each model. I will start with OpenAI using the model "gpt-5-nano."

In [ ]:
model_name = "gpt-5-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Next, I will start with the open-source models used by Ollama.

Ollama runs a local web service that gives an OpenAI compatible endpoint, and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

I will test the models:

llama 3.2

gemma 4

llama 2 uncensored



In [ ]:
# deepseek-r1:14b
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "deepseek-r1:14b"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# gemma 4
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "gemma4:latest"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# gpt-oss:latest
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "gpt-oss:latest"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# qwen3.5:latest
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "qwen3.5:latest"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

In [ ]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
#print(results)

In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")